In [1]:
import os
import torch
import utils
from transformers import AutoTokenizer

llmpath = utils.LLM_PATH['llama2-7bchat']
tokenizer = AutoTokenizer.from_pretrained(llmpath)
tokenizer.pad_token = tokenizer.eos_token
evaluator = utils.Eval(tokenizer=tokenizer, device='cpu')

text_list = utils.get_list_invert_text('chat_2m')
text_list = sorted(text_list, key=lambda x: len(tokenizer.encode(x, add_special_tokens=False)), reverse=True)
references_str = text_list
references_ids = [tokenizer.encode(x, add_special_tokens=False) for x in references_str]


Evaluation of ER and TBS

In [2]:
resultpath = '../results/white-box/short-prompt'

results = {'er':{}, 'tbs':{}}

results['er'][2] = torch.load(os.path.join(resultpath, 'chat_2m-0:479-er-l2-actmse-lr0.001-ep50000', 'invert-best.pt'), weights_only=True)['invert_text']
results['er'][4] = torch.load(os.path.join(resultpath, 'chat_2m-0:479-er-l4-actmse-lr0.001-ep50000', 'invert-best.pt'), weights_only=True)['invert_text']
results['er'][8] = torch.load(os.path.join(resultpath, 'chat_2m-0:200-er-l8-actmse-lr0.001-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                   torch.load(os.path.join(resultpath, 'chat_2m-200:479-er-l8-actmse-lr0.001-ep50000', 'invert-best.pt'), weights_only=True)['invert_text']

results['tbs'][4] = torch.load(os.path.join(resultpath, 'chat_2m-0:479-tbs-l4-actmse-lr0.001-ep50000', 'invert-best.pt'), weights_only=True)['invert_text']

results['tbs'][8] = torch.load(os.path.join(resultpath, 'chat_2m-0:170-tbs-l8-actmse-lr0.001-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                    torch.load(os.path.join(resultpath, 'chat_2m-170:479-tbs-l8-actmse-lr0.001-ep50000', 'invert-best.pt'), weights_only=True)['invert_text']

results['tbs'][16] = torch.load(os.path.join(resultpath, 'chat_2m-0:100-tbs-l16-actmse-lr0.001-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                    torch.load(os.path.join(resultpath, 'chat_2m-100:240-tbs-l16-actmse-lr0.001-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                    torch.load(os.path.join(resultpath, 'chat_2m-240:479-tbs-l16-actmse-lr0.001-ep50000', 'invert-best.pt'), weights_only=True)['invert_text']
results['tbs'][17] = torch.load(os.path.join(resultpath, 'chat_2m-0:100-tbs-l16-actmse-lr0.01-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                    torch.load(os.path.join(resultpath, 'chat_2m-100:240-tbs-l16-actmse-lr0.01-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                    torch.load(os.path.join(resultpath, 'chat_2m-240:479-tbs-l16-actmse-lr0.01-ep50000', 'invert-best.pt'), weights_only=True)['invert_text']

results['tbs'][24] = torch.load(os.path.join(resultpath, 'chat_2m-0:70-tbs-l24-actmse-lr0.01-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                    torch.load(os.path.join(resultpath, 'chat_2m-70:150-tbs-l24-actmse-lr0.01-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                    torch.load(os.path.join(resultpath, 'chat_2m-150:270-tbs-l24-actmse-lr0.01-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                    torch.load(os.path.join(resultpath, 'chat_2m-270:479-tbs-l24-actmse-lr0.01-ep50000', 'invert-best.pt'), weights_only=True)['invert_text']

results['tbs'][32] = torch.load(os.path.join(resultpath, 'chat_2m-0:40-tbs-l32-actmse-lr0.01-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                    torch.load(os.path.join(resultpath, 'chat_2m-40:90-tbs-l32-actmse-lr0.01-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                    torch.load(os.path.join(resultpath, 'chat_2m-90:140-tbs-l32-actmse-lr0.01-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                    torch.load(os.path.join(resultpath, 'chat_2m-140:200-tbs-l32-actmse-lr0.01-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                    torch.load(os.path.join(resultpath, 'chat_2m-200:280-tbs-l32-actmse-lr0.01-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                    torch.load(os.path.join(resultpath, 'chat_2m-280:360-tbs-l32-actmse-lr0.01-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                    torch.load(os.path.join(resultpath, 'chat_2m-360:479-tbs-l32-actmse-lr0.01-ep50000', 'invert-best.pt'), weights_only=True)['invert_text']

metrics = {}
for k, v in results.items():
    for k1, v1 in v.items():
        predictions_str = [x.replace('</s>', '') for x in v1]
        predictions_ids = [tokenizer.encode(x, add_special_tokens=False) for x in predictions_str]
        
        metrics[(k, k1)] = evaluator._text_comparison_metrics(predictions_ids, predictions_str, references_ids, references_str)

for k, v in metrics.items():
    print('Attack:', k[0], 'Layer:', k[1])
    print(f"{v['bge_emb_cos_sim_mean'] * 100:.2f}$\\pm$ {v['bge_emb_cos_sim_sem'] *100 :.2f} & {v['bleu_score']:.2f}$\\pm$ {v['bleu_score_sem']:.2f} & {v['rouge1_score']:.2f} & {v['exact_match'] * 100:.2f}$\\pm$ {v['exact_match_sem']:.2f} & {v['token_set_f1']*100:.2f} $\\pm$ {v['token_set_f1_sem']*100:.2f} ")


Attack: er Layer: 2
94.24$\pm$ 0.47 & 74.89$\pm$ 1.56 & 0.87 & 52.61$\pm$ 0.02 & 88.22 $\pm$ 0.90 
Attack: er Layer: 4
47.81$\pm$ 0.37 & 0.61$\pm$ 0.04 & 0.02 & 0.00$\pm$ 0.00 & 3.43 $\pm$ 0.21 
Attack: er Layer: 8
95.74$\pm$ 0.38 & 72.91$\pm$ 0.94 & 0.89 & 6.05$\pm$ 0.01 & 87.86 $\pm$ 0.69 
Attack: tbs Layer: 4
92.82$\pm$ 0.71 & 77.50$\pm$ 1.51 & 0.85 & 50.73$\pm$ 0.02 & 84.05 $\pm$ 1.33 
Attack: tbs Layer: 8
90.96$\pm$ 0.82 & 77.38$\pm$ 1.55 & 0.83 & 48.23$\pm$ 0.02 & 82.61 $\pm$ 1.44 
Attack: tbs Layer: 16
82.59$\pm$ 0.97 & 55.53$\pm$ 1.74 & 0.67 & 18.79$\pm$ 0.02 & 65.75 $\pm$ 1.73 
Attack: tbs Layer: 17
83.70$\pm$ 0.99 & 59.89$\pm$ 1.88 & 0.70 & 31.73$\pm$ 0.02 & 69.13 $\pm$ 1.72 
Attack: tbs Layer: 24
88.07$\pm$ 0.83 & 65.47$\pm$ 1.69 & 0.77 & 27.35$\pm$ 0.02 & 75.32 $\pm$ 1.53 
Attack: tbs Layer: 32
81.34$\pm$ 0.60 & 32.67$\pm$ 1.19 & 0.62 & 3.97$\pm$ 0.01 & 55.97 $\pm$ 1.02 


Evaluation of TS

In [3]:
resultpath = '../results/white-box/short-prompt'

results = {'ts':{}}

results['ts'][2] = torch.load(os.path.join(resultpath,'chat_2m-0:200-ts-l2-actmse-lr0.1-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                    torch.load(os.path.join(resultpath, 'chat_2m-200:479-ts-l2-actmse-lr0.1-ep50000', 'invert-best.pt'), weights_only=True)['invert_text']
results['ts'][8] = torch.load(os.path.join(resultpath,'chat_2m-0:180-ts-l8-actmse-lr0.01-ep50000', 'invert-best.pt'), weights_only=True)['invert_text'] + \
                    torch.load(os.path.join(resultpath, 'chat_2m-180:479-ts-l8-actmse-lr0.01-ep50000', 'invert-best.pt'), weights_only=True)['invert_text']

metrics = {}
for k, v in results.items():
    for k1, v1 in v.items():

        predictions_str = [x.replace('</s>', '') for x in v1]
        predictions_ids = [tokenizer.encode(x, add_special_tokens=False) for x in predictions_str]
        
        metrics[(k, k1)] = evaluator._text_comparison_metrics(predictions_ids, predictions_str, references_ids, references_str)
for k, v in metrics.items():
    print('Attack:', k[0], 'Layer:', k[1])
    print(f"{v['bge_emb_cos_sim_mean'] * 100:.2f}$\\pm$ {v['bge_emb_cos_sim_sem'] *100 :.2f} & {v['bleu_score']:.2f}$\\pm$ {v['bleu_score_sem']:.2f} & {v['rouge1_score']:.2f} & {v['exact_match'] * 100:.2f}$\\pm$ {v['exact_match_sem']:.2f} & {v['token_set_f1']*100:.2f} $\\pm$ {v['token_set_f1_sem']*100:.2f} ")


Attack: ts Layer: 2
39.34$\pm$ 0.22 & 0.00$\pm$ 0.00 & 0.00 & 0.00$\pm$ 0.00 & 0.00 $\pm$ 0.00 
Attack: ts Layer: 8
42.95$\pm$ 0.30 & 0.13$\pm$ 0.03 & 0.00 & 0.00$\pm$ 0.00 & 0.34 $\pm$ 0.08 
